# PoC: Embedding Version Migration + Rollback Pointer

Muc tieu notebook:
- Tao 2 embedding versions (`v1`, `v2`) cho cung tap chunks.
- Tao `index_manifest` va switch active pointer.
- Benchmark nhanh latency va recall proxy de quyet dinh rollback/cutover.

In [ ]:
import hashlib
import random
import statistics
import time
from datetime import datetime

random.seed(42)

In [ ]:
# 1) Sinh du lieu chunks gia lap (thay cho silver_chunks)
def make_chunks(n_chunks=1000):
    chunks = []
    for i in range(n_chunks):
        tenant_id = "tenant_a" if random.random() < 0.6 else "tenant_b"
        chunks.append(
            {
                "chunk_id": f"c_{i:05d}",
                "tenant_id": tenant_id,
                "document_id": f"doc_{random.randint(1, 120):04d}",
                "text": f"Dieu khoan hop dong so {i}",
                "token_count": random.randint(40, 260),
            }
        )
    return chunks


chunks = make_chunks(1000)
chunks[:3]

In [ ]:
# 2) Tao embeddings cho 2 model versions (fake embedding, stdlib only)
def l2_normalize(vec):
    norm = sum(x * x for x in vec) ** 0.5
    if norm == 0:
        return vec
    return [x / norm for x in vec]


def fake_embed(text, model_version, dim=32):
    digest = hashlib.sha256((text + model_version).encode("utf-8")).digest()
    raw = list(digest[:dim])
    vec = [float(x) for x in raw]
    return l2_normalize(vec)


def build_embedding_table(chunks_data, model_version):
    rows = []
    for row in chunks_data:
        rows.append(
            {
                "chunk_id": row["chunk_id"],
                "tenant_id": row["tenant_id"],
                "embedding_model_version": model_version,
                "embedding_vector": fake_embed(row["text"], model_version),
                "embed_ts": datetime.utcnow().isoformat(),
            }
        )
    return rows


emb_v1 = build_embedding_table(chunks, "v1")
emb_v2 = build_embedding_table(chunks, "v2")
silver_embeddings = emb_v1 + emb_v2
silver_embeddings[:2]

In [ ]:
# 3) Manifest quan ly active index pointer (thay cho gold_retrieval_index_manifest)
index_manifest = [
    {
        "index_id": "idx_tenant_a_v1",
        "tenant_id": "tenant_a",
        "model_version": "v1",
        "status": "active",
        "created_at": datetime.utcnow().isoformat(),
    },
    {
        "index_id": "idx_tenant_a_v2",
        "tenant_id": "tenant_a",
        "model_version": "v2",
        "status": "candidate",
        "created_at": datetime.utcnow().isoformat(),
    },
]


def get_active_version(manifest, tenant_id):
    for row in manifest:
        if row["tenant_id"] == tenant_id and row["status"] == "active":
            return row["model_version"]
    raise ValueError("No active index found")


def switch_active_index(manifest, tenant_id, to_version):
    out = [dict(row) for row in manifest]
    for row in out:
        if row["tenant_id"] == tenant_id:
            row["status"] = "inactive"
            if row["model_version"] == to_version:
                row["status"] = "active"
    return out


index_manifest

In [ ]:
# 4) Retrieval benchmark proxy: latency + recall@5
def cosine_sim(a, b):
    return sum(x * y for x, y in zip(a, b))


def percentile(values, p):
    if not values:
        return 0.0
    sorted_vals = sorted(values)
    k = int(round((p / 100.0) * (len(sorted_vals) - 1)))
    return sorted_vals[k]


def topk_indices(scores, k=5):
    idx_scores = list(enumerate(scores))
    idx_scores.sort(key=lambda x: x[1], reverse=True)
    return [i for i, _ in idx_scores[:k]]


def benchmark(manifest, tenant_id, n_queries=120):
    active_version = get_active_version(manifest, tenant_id)
    pool = [
        r
        for r in silver_embeddings
        if r["tenant_id"] == tenant_id and r["embedding_model_version"] == active_version
    ]
    vectors = [r["embedding_vector"] for r in pool]

    latencies = []
    recall_hits = 0

    for _ in range(n_queries):
        q_idx = random.randint(0, len(vectors) - 1)
        q = vectors[q_idx]

        t0 = time.perf_counter()
        scores = [cosine_sim(q, v) for v in vectors]
        top5 = topk_indices(scores, k=5)
        latencies.append((time.perf_counter() - t0) * 1000)

        if q_idx in top5:
            recall_hits += 1

    return {
        "tenant_id": tenant_id,
        "active_version": active_version,
        "p95_latency_ms": round(percentile(latencies, 95), 4),
        "avg_latency_ms": round(statistics.mean(latencies), 4),
        "recall_at_5_proxy": round(recall_hits / n_queries, 4),
    }


baseline = benchmark(index_manifest, "tenant_a", n_queries=120)
baseline

In [ ]:
# 5) Cutover sang v2, benchmark, va rollback neu quality kem
candidate_manifest = switch_active_index(index_manifest, "tenant_a", "v2")
candidate = benchmark(candidate_manifest, "tenant_a", n_queries=120)

print("Baseline:", baseline)
print("Candidate:", candidate)
print("Latency delta (ms):", round(candidate["p95_latency_ms"] - baseline["p95_latency_ms"], 4))
print("Recall delta:", round(candidate["recall_at_5_proxy"] - baseline["recall_at_5_proxy"], 4))

recall_drop = baseline["recall_at_5_proxy"] - candidate["recall_at_5_proxy"]
if recall_drop > 0.03:
    final_manifest = switch_active_index(candidate_manifest, "tenant_a", "v1")
    decision = f"ROLLBACK to v1 (recall drop={recall_drop:.4f})"
else:
    final_manifest = candidate_manifest
    decision = "KEEP v2 as active"

print("Decision:", decision)
print("Final manifest:")
for row in final_manifest:
    print(row)